# Frozen checkpoints — full five-seed experiment
Run **all cells once**, from `notebooks/medium_world` on `experiment/phase-aware-lr`.
Reproduce **Constant LR**, 5 seeds × 2 curricula × 9,000 updates = **10 full training runs**; save checkpoints at steps 5,400, 7,200 and 9,000 (**30 model checkpoints**).

No pilot configuration. Existing complete checkpoint groups are reused only after provenance validation. An incomplete group causes a clear error instead of silently blending incompatible checkpoints. Training uses the same initialization, augmentation, schedule, AdamW, clipping and evaluation calls as `lr_reset.ipynb`. It does **not** alter old experiment artifacts.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEEDS = (42, 43, 44, 45, 46)
SCHEDULES = ('prereq_mixed_pair_focus', 'mixed')
STEPS = (5400, 7200, 9000)
TOTAL_STEPS = 9000
BATCH_SIZE = 50
EVAL_EVERY = 100
DEVICE = 'cuda'
DATA_DIR = PROJECT_ROOT / 'outputs/medium_world/dataset'
OUT_DIR = PROJECT_ROOT / 'outputs/medium_world/experiments/representation_diagnostics/five_seeds'
ARCHIVE_DIR = PROJECT_ROOT / 'outputs/medium_world/experiments/phase_aware_lr'
EXPECTED_HASHES = {
    'train.jsonl': '69deccacbac6db25f7ffc8165f01545483d856855282ea640997f6660ce27c26',
    'id_dev.jsonl': 'ece4b831f1f7fe6efad42b18aca499f12c2d573baa76fc218864bb56fe24b11a',
    'ood_dev.jsonl': '208a7566cc2082fa4438ec0d003430d0927c54bf82c55bb40fa885ef4961e0aa'
}

In [2]:
from collections import Counter
import gc
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from src.medium_visual_world.learning import (
    TASKS, MediumDataset, AttributeBindingViT,
    augment_images, set_seed, environment,
)
from src.medium_visual_world.scaffolded_spiral import make_spiral_blocks
from src.medium_visual_world.lr_policies import LRPolicy, effective_phases, evaluation_steps
from src.medium_visual_world.prediction_tracking import evaluate_predictions, load_trace, step_index
from src.medium_visual_world.representation_probing import sha256_file, data_hash_audit
assert torch.cuda.is_available() or DEVICE == 'cpu', 'Use DEVICE=cpu if CUDA unavailable'
device = torch.device(DEVICE)
torch.set_num_threads(min(4, torch.get_num_threads()))
OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_hashes = data_hash_audit(DATA_DIR, EXPECTED_HASHES)
CODE_FILES = {
    'learning': 'src/medium_visual_world/learning.py',
    'scaffolded_spiral': 'src/medium_visual_world/scaffolded_spiral.py',
    'lr_policies': 'src/medium_visual_world/lr_policies.py',
    'render': 'src/medium_visual_world/render.py',
    'world': 'src/medium_visual_world/world.py',
    'prediction_tracking': 'src/medium_visual_world/prediction_tracking.py',
}
code_hashes = {
    name: sha256_file(PROJECT_ROOT / path)
    for name, path in CODE_FILES.items()
}
train_data = MediumDataset(DATA_DIR / 'train.jsonl')
id_dev = MediumDataset(DATA_DIR / 'id_dev.jsonl')
ood_dev = MediumDataset(DATA_DIR / 'ood_dev.jsonl')
assert len(train_data) == 10000 and len(id_dev) == 1024 and (len(ood_dev) == 400)
assert len({r['scene_id'] for r in train_data.rows} & {r['scene_id'] for r in id_dev.rows}) == 0
print('Runs', len(SEEDS) * len(SCHEDULES), '| Expected checkpoints', len(SEEDS) * len(SCHEDULES) * len(STEPS))
print('Device', environment(), '| Dataset hashes', manifest_hashes)

Runs 10 | Expected checkpoints 30
Device {'python': '3.12.4', 'torch': '2.13.0+cu130', 'cuda_build': '13.0', 'cuda_available': True, 'gpu': 'NVIDIA GeForce RTX 5050 Laptop GPU'} | Dataset hashes {'train.jsonl': '69deccacbac6db25f7ffc8165f01545483d856855282ea640997f6660ce27c26', 'id_dev.jsonl': 'ece4b831f1f7fe6efad42b18aca499f12c2d573baa76fc218864bb56fe24b11a', 'ood_dev.jsonl': '208a7566cc2082fa4438ec0d003430d0927c54bf82c55bb40fa885ef4961e0aa'}


## Reproduce the original constant-LR training process
Only model `state_dict`s and provenance are stored. These are diagnostic inference checkpoints, **not** resumable optimizer checkpoints. Archived prediction traces are compared after the runs, by exact question ID and checkpoint step.

In [3]:
def load_diagnostic_checkpoint(path):
    # Older checkpoints saved torch.__version__ as the TorchVersion str subclass.
    # Allowlist precisely that type while keeping restricted weights-only loading.
    with torch.serialization.safe_globals([torch.torch_version.TorchVersion]):
        return torch.load(path, map_location='cpu', weights_only=True)

def checkpoint_path(schedule, seed, step):
    return OUT_DIR / 'checkpoints' / f'{schedule}_constant_seed_{seed}_step_{step}.pt'

def checkpoint_record(path, schedule, seed, step):
    saved = load_diagnostic_checkpoint(path)
    assert saved['schedule'] == schedule and saved['seed'] == seed and (saved['step'] == step)
    assert saved['dataset_hashes'] == manifest_hashes and saved['code_hashes'] == code_hashes
    assert saved['model_config'] == {'dim': 48, 'patch_size': 4, 'depth': 1}
    assert saved['training_config'] == {
        'total_steps': 9000,
        'batch_size': 50,
        'lr': 0.0001,
        'weight_decay': 0.0,
        'clip_norm': 1.0,
        'augmentation': True
    }
    assert all((t.isfinite().all() for t in saved['model'].values()))
    return {
        'schedule': schedule,
        'seed': seed,
        'step': step,
        'checkpoint_sha256': sha256_file(path),
        'path': str(path)
    }

def train_one(schedule, seed):
    paths = {step: checkpoint_path(schedule, seed, step) for step in STEPS}
    found = [path.is_file() for path in paths.values()]
    if all(found):
        print('Reusing verified complete group:', schedule, seed)
        return [checkpoint_record(path, schedule, seed, step) for step, path in paths.items()]
    if any(found):
        # A model-only checkpoint cannot resume this run exactly: the optimizer
        # and RNG states were not saved. Preserve every existing checkpoint,
        # then restart ONLY this incomplete schedule/seed from step zero.
        base = OUT_DIR / 'incomplete_checkpoint_backups'
        backup = base / f'{schedule}_seed_{seed}'
        version = 1
        while backup.exists():
            backup = base / f'{schedule}_seed_{seed}_v{version}'
            version += 1
        backup.mkdir(parents=True, exist_ok=False)
        for checkpoint in paths.values():
            if checkpoint.is_file():
                checkpoint.replace(backup / checkpoint.name)
        print(
            f'Archived incomplete checkpoints: {backup}. Restarting {schedule}, seed {seed} from step zero.',
            flush=True
        )
    set_seed(seed)
    model = AttributeBindingViT(dim=48, patch_size=4, depth=1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.0)
    blocks, plan = make_spiral_blocks(train_data.rows, schedule, seed)
    phases = effective_phases(plan['phases'])
    controller = LRPolicy('constant', optimizer, phases, initial_lr=0.0001, final_mixed_lr=5e-05, min_lr=1e-06, factor=2 / 3, patience=2, threshold=0.001, check_every=EVAL_EVERY)
    eval_steps = set(evaluation_steps(plan['phases'], phases, EVAL_EVERY))
    seen = Counter()
    step = 0
    records = []

    def evaluate_step():
        for split, data in (('id_dev', id_dev), ('ood_dev', ood_dev)):
            result, _ = evaluate_predictions(model, data, device, BATCH_SIZE)
            if split == 'id_dev':
                id_loss = result['loss']
        controller.observe(step, id_loss)
    # Match the original run's step-zero development evaluation.
    evaluate_step()

    for phase, indices in blocks:
        for start in range(0, len(indices), BATCH_SIZE):
            ids = indices[start:start + BATCH_SIZE]
            x, q, y = train_data.batch(ids, device)
            x = augment_images(x)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            controller.before_update(step)
            loss = nn.functional.cross_entropy(model(x, q), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1
            seen.update((train_data.tasks[i] for i in ids))
            if step in eval_steps:
                evaluate_step()
            if step in paths:
                path = paths[step]
                path.parent.mkdir(parents=True, exist_ok=True)
                obj = {
                    'model': {
                        key: value.detach().cpu().clone()
                        for key, value in model.state_dict().items()
                    },
                    'schedule': schedule,
                    'seed': seed,
                    'step': step,
                    'dataset_hashes': manifest_hashes,
                    'code_hashes': code_hashes,
                    'model_config': {'dim': 48, 'patch_size': 4, 'depth': 1},
                    'training_config': {
                        'total_steps': 9000,
                        'batch_size': 50,
                        'lr': 0.0001,
                        'weight_decay': 0.0,
                        'clip_norm': 1.0,
                        'augmentation': True
                    },
                    'environment': {
                        key: str(value) if isinstance(value, str) else value
                        for key, value in environment().items()
                    }
                }
                temp = path.with_suffix('.tmp')
                torch.save(obj, temp)
                temp.replace(path)
                records.append(checkpoint_record(path, schedule, seed, step))
                print('Saved', schedule, 'seed', seed, 'step', step, flush=True)
    assert step == TOTAL_STEPS and set(seen) == set(TASKS)
    assert all((seen[task] == 112500 for task in TASKS)), seen
    assert len(records) == len(STEPS)
    del model, optimizer
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return records
manifest = []
for schedule in SCHEDULES:
    for seed in SEEDS:
        manifest.extend(train_one(schedule, seed))
manifest = pd.DataFrame(manifest).sort_values(['schedule', 'seed', 'step'])
assert len(manifest) == 30 and (not manifest.duplicated(['schedule', 'seed', 'step']).any())
manifest.to_csv(OUT_DIR / 'checkpoint_manifest.csv', index=False)
display(manifest)

Archived incomplete checkpoints: C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AIO Conquer 5 - visual_world\The-order-of-Learning\outputs\medium_world\experiments\representation_diagnostics\five_seeds\incomplete_checkpoint_backups\prereq_mixed_pair_focus_seed_42_v1. Restarting prereq_mixed_pair_focus, seed 42 from step zero.
Saved prereq_mixed_pair_focus seed 42 step 5400
Saved prereq_mixed_pair_focus seed 42 step 7200
Saved prereq_mixed_pair_focus seed 42 step 9000
Saved prereq_mixed_pair_focus seed 43 step 5400
Saved prereq_mixed_pair_focus seed 43 step 7200
Saved prereq_mixed_pair_focus seed 43 step 9000
Saved prereq_mixed_pair_focus seed 44 step 5400
Saved prereq_mixed_pair_focus seed 44 step 7200
Saved prereq_mixed_pair_focus seed 44 step 9000
Saved prereq_mixed_pair_focus seed 45 step 5400
Saved prereq_mixed_pair_focus seed 45 step 7200
Saved prereq_mixed_pair_focus seed 45 step 9000
Saved prereq_mixed_pair_focus seed 46 step 5400
Saved prereq_mixed_pair_focus seed 46 step 7200
Sav

,schedule,seed,step,checkpoint_sha256,path
15,mixed,42,5400,1a5f298404afbfe33c736a02a8b49b4fb77aab7bcb62ce...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
16,mixed,42,7200,eb3e098b5ac9ef520d93d7359a4b74b79b806de5745fcd...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
17,mixed,42,9000,ce6a40dcda1c502ace8a3bfaec368b79772e00796e01f6...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
18,mixed,43,5400,c926ae40f795bbe0c54a3fa3bb60f52f975e4dad418728...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
19,mixed,43,7200,8d0dcc9706e4abbf5b01a93d5de53a914f89c4841f3e3e...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
20,mixed,43,9000,a3c7e979fcef434e6b949f7215b64b0d1ea02f394a552e...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
21,mixed,44,5400,80339ce336bf688d27020d049fcde4698d0fa367033605...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
22,mixed,44,7200,ca2063a0616c880de7ac3176c0dd9a8aed1f405afc5bee...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
23,mixed,44,9000,7387a9f1a497e2eeb5e669ea0aeb5fe57b7fe97896eb43...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
24,mixed,45,5400,4251f7d5185c097270a625ba673764bcaa32c901a91845...,C:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...


## Compare reproduced models with archived LR-reset traces
The comparison is diagnostic rather than a requirement for bitwise equality. Any mismatches or missing archived traces remain explicit; **do not silently call these the original trained weights**.

In [4]:
audit = []
original_scores = []
for record in manifest.to_dict('records'):
    schedule, seed, step = (record['schedule'], int(record['seed']), int(record['step']))
    saved = load_diagnostic_checkpoint(record['path'])
    model = AttributeBindingViT(**saved['model_config']).to(device).eval()
    model.load_state_dict(saved['model'], strict=True)
    trace_path = ARCHIVE_DIR / 'prediction_traces' / f'{schedule}_constant_seed_{seed}.npz'
    old = load_trace(trace_path) if trace_path.is_file() else None
    if old is not None and old['metadata'].get('dataset_hashes') != manifest_hashes:
        raise AssertionError(f'Archived trace dataset mismatch: {trace_path}')
    for split, data in (('id_dev', id_dev), ('ood_dev', ood_dev)):
        summary, pred = evaluate_predictions(model, data, device, BATCH_SIZE)
        for task, value in summary['by_template'].items():
            original_scores.append({
                'schedule': schedule,
                'seed': seed,
                'step': step,
                'split': split,
                'task': task,
                'accuracy': value,
                'cross_entropy': summary['loss_by_template'][task]
            })
        row = {**record, 'split': split, 'n_questions': len(data)}
        if old is None:
            row['status'] = 'archive_missing'
            row['mismatched_predictions'] = None
            row['max_probability_delta'] = None
        else:
            block = old['splits'][split]
            j = step_index(block, step)
            assert block['ids'].tolist() == [str(r['id']) for r in data.rows]
            row['status'] = 'compared'
            row['mismatched_predictions'] = int(np.count_nonzero(pred['predicted_label'] != block['predictions'][j]))
            row['max_probability_delta'] = float(np.max(np.abs(pred['probability_yes'] - block['probabilities'][j])))
        audit.append(row)
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
audit = pd.DataFrame(audit)
audit.to_csv(OUT_DIR / 'checkpoint_reproduction_audit.csv', index=False)
pd.DataFrame(original_scores).to_csv(OUT_DIR / 'original_checkpoint_performance.csv', index=False)
assert len(audit) == 60 and len(original_scores) > 0
display(audit[['schedule', 'seed', 'step', 'split', 'status', 'mismatched_predictions', 'max_probability_delta']])
print('Checkpoint phase complete:', len(manifest), 'checkpoints and', len(audit), 'reproduction comparisons')

,schedule,seed,step,split,status,mismatched_predictions,max_probability_delta
0,mixed,42,5400,id_dev,compared,0,0.000000
1,mixed,42,5400,ood_dev,compared,0,0.000000
2,mixed,42,7200,id_dev,compared,0,0.000000
3,mixed,42,7200,ood_dev,compared,0,0.000000
4,mixed,42,9000,id_dev,compared,0,0.000000
5,mixed,42,9000,ood_dev,compared,0,0.000000
6,mixed,43,5400,id_dev,compared,0,0.000000
7,mixed,43,5400,ood_dev,compared,0,0.000000
8,mixed,43,7200,id_dev,compared,0,0.000000
9,mixed,43,7200,ood_dev,compared,0,0.000000


Checkpoint phase complete: 30 checkpoints and 60 reproduction comparisons
